# Constituency Parsing: The Syntactic Architect

**Project Brief**

Welcome, **Chief Architect**.

Language is not a flat stream of bricks (words). It is a structured edifice. Words group together to form stable components (**Constituents**), which support larger structures, ultimately forming the complete building (Sentence).

Your Job:
1.  **Analyze Materials**: Identify the constituents (NPs, VPs, PPs).
2.  **Draft Blueprints**: Write **Context-Free Grammars (CFGs)** to define allowed structures.
3.  **Inspect Stability**: Resolve Structural Ambiguity.
4.  **Standardize**: Convert grammars to **Chomsky Normal Form (CNF)**.
5.  **Build the Foundation**: Implement the **CKY Algorithm** to parse sentences bottom-up.

---

In [1]:
# Architectural Tools (Setup)
!pip install nltk pandas numpy svgling

import nltk
import pandas as pd
import numpy as np
from nltk import Tree
from nltk.draw.tree import TreeView

# Download the Blueprint Archive
nltk.download('treebank')


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


/Users/rajaramkankipati/Documents/GitHub/AI_academy/.venv/lib/python3.11/site-packages/nltk/draw/__init__.py:15: UserWarning: nltk.draw package not loaded (please install Tkinter library).
  warnings.warn("nltk.draw package not loaded (please install Tkinter library).")


ModuleNotFoundError: No module named '_tkinter'

## 1. Constituency (The Material Blocks)

Words clump together. A "Noun Phrase" (NP) acts like a single Noun. A "Verb Phrase" (VP) acts like a single Verb.

**Stress Tests** (How to prove a group of words is a constituent):
1.  **Substitution**: Can you replace it with a single pronoun? ("The tall man" -> "He")
2.  **Movement**: Can you move the whole group to the front? ("I like [the red apple]" -> "[The red apple] is what I like")
3.  **Coordination**: Can you join it with a similar group? ("[The dog] and [the cat]")

In [2]:
# Visualizing a Structure
structure = "(S (NP (Det The) (N Architect)) (VP (V built) (NP (Det a) (N skyscraper))))"
tree = Tree.fromstring(structure)
print(tree)
# tree.pretty_print() # ASCII Art blueprint

(S
  (NP (Det The) (N Architect))
  (VP (V built) (NP (Det a) (N skyscraper))))


## 2. Context-Free Grammars (The Blueprints)

A **CFG** is a set of rules describing how to build valid structures.
*   **Terminals**: The words (Bricks).
*   **Non-Terminals**: The categories (Walls, Floors, Rooms) - e.g., NP, VP, S.
*   **Rules**: $A \rightarrow B \ C$ (A consists of B and C).

In [3]:
# Drafting a Grammar
grammar_text = """
  S -> NP VP
  NP -> Det N | Det N PP
  VP -> V NP | V NP PP
  PP -> P NP
  Det -> 'the' | 'a'
  N -> 'man' | 'telescope' | 'park'
  V -> 'saw'
  P -> 'in' | 'with'
"""
grammar = nltk.CFG.fromstring(grammar_text)

parser = nltk.ChartParser(grammar)
sentence = "the man saw a park with a telescope".split()

print("Valid Structures:")
for tree in parser.parse(sentence):
    print(tree)

Valid Structures:
(S
  (NP (Det the) (N man))
  (VP
    (V saw)
    (NP (Det a) (N park))
    (PP (P with) (NP (Det a) (N telescope)))))
(S
  (NP (Det the) (N man))
  (VP
    (V saw)
    (NP (Det a) (N park) (PP (P with) (NP (Det a) (N telescope))))))


## 3. Ambiguity (Structural Instability)

"I saw the man with the telescope."
Did I use the telescope to see him? Or did he have the telescope?

**Structural Ambiguity** is when one string of terminals supports multiple valid non-terminal trees. This is dangerous – it means the blueprint can be interpreted in two ways.

### Case A: VP Attachment (Instrument)
"I [saw [the man] [with the telescope]]"
The PP attaches to the VP. The seeing happened with the telescope.

### Case B: NP Attachment (Possession)
"I [saw [the man [with the telescope]]]"
The PP attaches to the Object NP. The man possesses the telescope.

**Recursion**: Note how `NP -> Det N PP` and `PP -> P NP` allows for infinite nesting. "The cat in the hat on the mat..." is mathematically valid!

## 4. Normal Forms (Standardization)

To build efficient parsers (like CKY), we need standardized blueprints.
**Chomsky Normal Form (CNF)** requires all rules to be:
1.  $A \rightarrow B \ C$ (Binary Non-terminals)
2.  $A \rightarrow w$ (Single Terminal)

No mixing ($A \rightarrow B \ w$), no singles ($A \rightarrow B$), no triples ($A \rightarrow B \ C \ D$).

In [4]:
# Converting to CNF
# NLTK can handle this conversion automatically for us, usually.
# But let's verify our manual logic.
# Rule: VP -> V NP PP
# Becomes:
# VP -> V X1
# X1 -> NP PP


## 5. CKY Parsing (The Foundation Inspection)

The **Cocke-Younger-Kasami (CKY)** algorithm is a bottom-up dynamic programming approach.
It builds a triangular chart. Cell $(i, j)$ contains all Non-Terminals that span from word $i$ to $j$.

### Manual Inspection (Paper Trace)
Let's trace: "the dog chased a cat" (Length 5)
Words: `[0] the [1] dog [2] chased [3] a [4] cat [5]`

**Level 1 (Diagonals)**:
*   Span (0,1) 'the' -> `Det`
*   Span (1,2) 'dog' -> `N` -> `NP` (Unaries handled)
*   Span (2,3) 'chased' -> `V`

**Level 2**:
*   Span (0,2) 'the dog' ($Det + N$) -> `NP`

**Level 3, 4...**
Let's look at Span (2, 5) "chased a cat".
*   Split at 3:
    *   (2,3) 'chased' = `V`
    *   (3,5) 'a cat' = `NP`
    *   Rule `VP -> V NP`?
    *   YES! Add `VP` to cell (2, 5).

In [5]:
def cky_parse(words, grammar):
    # 1. Convert NLTK grammar to a friendly lookup
    # terminals[word] = [list of LHS]
    # binary[(B, C)] = [list of LHS]
    terminals = {}
    binary = {}
    
    for production in grammar.productions():
        lhs = production.lhs()
        rhs = production.rhs()
        if len(rhs) == 1 and isinstance(rhs[0], str):
            # Terminal rule A -> w
            word = rhs[0]
            if word not in terminals: terminals[word] = []
            terminals[word].append(lhs)
        elif len(rhs) == 2:
            # Binary rule A -> B C
            key = (rhs[0], rhs[1])
            if key not in binary: binary[key] = []
            binary[key].append(lhs)
            
    n = len(words)
    # Chart: (n+1) x (n+1) table. 
    # table[i][j] stores a SET of Non-Terminals defining span i:j
    table = [[set() for _ in range(n + 1)] for _ in range(n + 1)]
    
    # 2. Initialization (Diagonal)
    for i in range(n):
        word = words[i]
        if word in terminals:
            for lhs in terminals[word]:
                table[i][i+1].add(lhs)
                
    # 3. Main Loop
    for length in range(2, n + 1): # Span length
        for i in range(n - length + 1): # Start index
            j = i + length # End index
            for k in range(i + 1, j): # Split point
                # Look for A -> B C spanning (i,k) and (k,j)
                left_cell = table[i][k]
                right_cell = table[k][j]
                
                for B in left_cell:
                    for C in right_cell:
                        if (B, C) in binary:
                            for A in binary[(B, C)]:
                                table[i][j].add(A)
                                
    return table

# Test it
grammar_cnf = nltk.CFG.fromstring("""
  S -> NP VP
  VP -> V NP
  NP -> Det N
  Det -> 'the' | 'a'
  N -> 'dog' | 'cat'
  V -> 'chased'
""")

words = "the dog chased a cat".split()
chart = cky_parse(words, grammar_cnf)

# Check top right cell for 'S'
n = len(words)
print(f"Did we build an S? {grammar_cnf.start() in chart[0][n]}")

# Visualize Chart
print("Chart Contents:")
for i in range(n):
    for j in range(i+1, n+1):
        if chart[i][j]:
            print(f"Span [{i}:{j}] '{' '.join(words[i:j])}': {chart[i][j]}")

Did we build an S? True
Chart Contents:
Span [0:1] 'the': {Det}
Span [0:2] 'the dog': {NP}
Span [0:5] 'the dog chased a cat': {S}
Span [1:2] 'dog': {N}
Span [2:3] 'chased': {V}
Span [2:5] 'chased a cat': {VP}
Span [3:4] 'a': {Det}
Span [3:5] 'a cat': {NP}
Span [4:5] 'cat': {N}


## 6. Treebanks (The Blueprint Archive)

Real architects don't write rules from scratch. They study existing buildings.
The **Penn Treebank** is a massive collection of parsed Wall Street Journal articles.

In [6]:
from nltk.corpus import treebank

print(f"Total Parsed Sentences: {len(treebank.parsed_sents())}")
sample_tree = treebank.parsed_sents()[0]
print("Sample Tree:")
print(sample_tree)

Total Parsed Sentences: 3914
Sample Tree:
(S
  (NP-SBJ
    (NP (NNP Pierre) (NNP Vinken))
    (, ,)
    (ADJP (NP (CD 61) (NNS years)) (JJ old))
    (, ,))
  (VP
    (MD will)
    (VP
      (VB join)
      (NP (DT the) (NN board))
      (PP-CLR (IN as) (NP (DT a) (JJ nonexecutive) (NN director)))
      (NP-TMP (NNP Nov.) (CD 29))))
  (. .))


## 7. Evaluation (Safety Certification)

How do we know if our parser matches the official blueprints?
We use **PARSEVAL** metrics:
*   **Precision**: How many of my suggested constituents are correct?
*   **Recall**: How many of the true constituents did I find?
*   **Crossing Brackets**: Do my walls cross the existing structural lines?

In [7]:
# Example of comparing trees (Conceptual)
gold_standard = Tree.fromstring("(S (NP I) (VP (V saw) (NP him)))")
my_parse = Tree.fromstring("(S (NP I) (VP (V saw) (NP him)))")

print(f"Match: {gold_standard == my_parse}")

Match: True
